In [1]:
!pip install -U sentence-transformers

In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset

/Users/rayyanzaid/Desktop/School/CSCI-566-DeepLearning/CSCI-566-Course-Project-DeepPrep-AI/csci-566-project-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def parse_transcript(transcript_str):
    """Parse '[00:01 - 00:11] text' lines into list of (timestamp, text)."""
    if not transcript_str or not transcript_str.strip():
        return []
    segments = []
    for line in transcript_str.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("[") and "]" in line:
            idx = line.index("]")
            timestamp = line[1:idx].strip()
            text = line[idx + 1 :].strip()
            if text:
                segments.append((timestamp, text))
        else:
            segments.append(("", line))
    return segments


In [ ]:
import librosa
import numpy as np
from moviepy import VideoFileClip
import tempfile
import numpy as np

# Feature Extraction Functions

# 1. Energy and Power
def extract_energy(speech):
    return librosa.feature.rms(y=speech)[0]  # Root mean square energy

# 2. Pitch (Fundamental Frequency) Statistics
def extract_pitch(speech, rate):
    pitch, _ = librosa.core.piptrack(y=speech, sr=rate)
    pitch = pitch[pitch > 0]  # Remove zero values
    return pitch

# 3. Pitch Statistics
def pitch_stats(pitch):
    # If pitch is empty, return zeros for all features
    if pitch is None or len(pitch) == 0:
        return (
            0.0,                     # min_pitch
            0.0,                     # max_pitch
            0.0,                     # mean_pitch
            0.0,                     # pitch_sd
            0.0,                     # pitch_abs
            np.array([0.0, 0.0, 0.0]),  # pitch_quant
            0.0,                     # diff_pitch_max_min
            0.0,                     # diff_pitch_max_mean
            0.0                      # diff_pitch_max_mode
        )

    # Normal computation if pitch has values
    min_pitch = np.min(pitch)
    max_pitch = np.max(pitch)
    mean_pitch = np.mean(pitch)
    pitch_sd = np.std(pitch)
    pitch_abs = np.mean(np.abs(pitch))
    pitch_quant = np.quantile(pitch, [0.25, 0.5, 0.75])
    diff_pitch_max_min = max_pitch - min_pitch
    diff_pitch_max_mean = max_pitch - mean_pitch
    diff_pitch_max_mode = max_pitch - np.median(pitch)

    return min_pitch, max_pitch, mean_pitch, pitch_sd, pitch_abs, pitch_quant, diff_pitch_max_min, diff_pitch_max_mean, diff_pitch_max_mode

# 4. Intensity (RMS)
def intensity_features(speech):
    intensity = librosa.feature.rms(y=speech)[0]
    intensity_min = np.min(intensity)
    intensity_max = np.max(intensity)
    intensity_mean = np.mean(intensity)
    intensity_sd = np.std(intensity)
    intensity_quant = np.quantile(intensity, [0.25, 0.5, 0.75])
    diff_int_max_min = intensity_max - intensity_min
    diff_int_max_mean = intensity_max - intensity_mean
    diff_int_max_mode = intensity_max - np.median(intensity)
    return intensity_min, intensity_max, intensity_mean, intensity_sd, intensity_quant, diff_int_max_min, diff_int_max_mean, diff_int_max_mode

# 5. Jitter and Shimmer
def jitter_shimmer(speech, rate):
    # Jitter: Measure of pitch variation
    pitch, voiced_flag = librosa.core.piptrack(y=speech, sr=rate)
    jitter = np.std(pitch[pitch > 0])
    
    # Shimmer: Measure of amplitude variation (calculated based on RMS)
    intensity = librosa.feature.rms(y=speech)[0]
    shimmer = np.std(intensity)
    
    return jitter, shimmer

# 6. Speech Rate and Pauses
def speech_rate(speech, rate):
    onset_env = librosa.onset.onset_strength(y=speech, sr=rate)
    onset_frames = librosa.onset.onset_detect(onset_envelope=onset_env, sr=rate)
    speak_rate = len(onset_frames) / (len(speech) / rate)  # Onsets per second
    return speak_rate

def pause_features(speech, rate):
    onset_env = librosa.onset.onset_strength(y=speech, sr=rate)
    onset_frames = librosa.onset.onset_detect(onset_envelope=onset_env, sr=rate)
    intervals = librosa.frames_to_time(onset_frames, sr=rate)  # Time intervals between onsets
    pauses = np.diff(intervals)  # Calculate pause durations
    max_pause = np.max(pauses) if len(pauses) > 0 else 0
    avg_pause = np.mean(pauses) if len(pauses) > 0 else 0
    total_pause_duration = np.sum(pauses)
    return max_pause, avg_pause, total_pause_duration

# 7. Rising and Falling Edges (Pitch changes)
def rising_falling_edges(pitch):
    rising = np.sum(np.diff(pitch) > 0)
    falling = np.sum(np.diff(pitch) < 0)
    max_rising = np.max(np.diff(pitch)[np.diff(pitch) > 0]) if len(np.diff(pitch)) > 0 else 0
    max_falling = np.min(np.diff(pitch)[np.diff(pitch) < 0]) if len(np.diff(pitch)) > 0 else 0
    avg_rise = np.mean(np.diff(pitch)[np.diff(pitch) > 0]) if len(np.diff(pitch)) > 0 else 0
    avg_fall = np.mean(np.diff(pitch)[np.diff(pitch) < 0]) if len(np.diff(pitch)) > 0 else 0
    return rising, falling, max_rising, max_falling, avg_rise, avg_fall

# 8. Loudness (RMS energy)
def loudness(speech):
    return librosa.feature.rms(y=speech)[0]



# Example of timestamp_ranges: [(0, 5), (10, 15), (20, 25)] representing segments of the video to analyze

def analyze_prosody(video_path, timestamp_ranges):

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_audio_file:
        audio_path = tmp_audio_file.name
        clip = VideoFileClip(video_path)
        if clip.audio is None:
            # No audio in the video → return a single row of zeros
            return [np.zeros((1, 13))]
        clip.audio.write_audiofile(audio_path)

    speech, rate = librosa.load(audio_path, sr=None)

    prosody_features_2D_array = []

    for start, end in timestamp_ranges:
        start_sample = int(start * rate)
        end_sample = int(end * rate)
        segment_speech = speech[start_sample:end_sample]

        # --- ADD CHECK FOR EMPTY SEGMENTS ---
        if len(segment_speech) == 0:
            prosody_features_2D_array.append(np.zeros((1, 13)))
            continue

        energy = extract_energy(segment_speech)
        pitch = extract_pitch(segment_speech, rate)
        min_pitch, max_pitch, mean_pitch, pitch_sd, pitch_abs, pitch_quant, diff_pitch_max_min, diff_pitch_max_mean, diff_pitch_max_mode = pitch_stats(pitch)
        intensity_min, intensity_max, intensity_mean, intensity_sd, intensity_quant, diff_int_max_min, diff_int_max_mean, diff_int_max_mode = intensity_features(segment_speech)
        jitter, shimmer = jitter_shimmer(segment_speech, rate)

        # --- SAFE SPEECH RATE CALCULATION ---
        try:
            speak_rate = speech_rate(segment_speech, rate)
        except ZeroDivisionError:
            speak_rate = 0.0

        max_pause, avg_pause, total_pause_duration = pause_features(segment_speech, rate)

        feature_vector_at_each_time = np.array([
            np.mean(energy),
            min_pitch,
            max_pitch,
            mean_pitch,
            pitch_sd,
            intensity_min,
            intensity_max,
            intensity_mean,
            jitter,
            shimmer,
            speak_rate,
            max_pause,
            avg_pause
        ]).reshape(1, -1)

        prosody_features_2D_array.append(feature_vector_at_each_time)

    return prosody_features_2D_array







In [19]:

"""

Purpose: This function will go through the timestamp ranges in transcript_segments and compute the prosody features for the corresponding segments of the audio in the video.

Input: Video Path & Transcript Segments

    Example of transcript_segments:
    {
    "0:01 - 0:11": "Hello, my name is Rayyan.",
    "0:12 - 0:20": "Harish is working on this model with me.",
    }

Output: A 2D numpy array of shape (num_transcript_segments, num_prosody_features)
    Each row - corresponds to a transcript_segment (a timestamp range) 
    Each column - corresponds to a specific prosody feature (e.g., pitch, energy, speaking rate, etc.)

    Example of output array:
    [
        

    
    ]
"""
def parse_timestamp_range(ts_string):
    """
    Converts "0:01 - 0:11" → (1.0, 11.0)
    """
    start_str, end_str = ts_string.split(" - ")

    def to_seconds(t):
        parts = t.split(":")
        if len(parts) == 2:
            minutes, seconds = parts
            return int(minutes) * 60 + float(seconds)
        elif len(parts) == 3:
            hours, minutes, seconds = parts
            return int(hours)*3600 + int(minutes)*60 + float(seconds)

    return (to_seconds(start_str), to_seconds(end_str))

def parse_audio_return_prosody_features(video_path, transcript_segments)-> np.ndarray:
    # print(f"Video Path: {video_path}")
    # print(f"Transcript Segments: {transcript_segments}")

    # Get the timestamp ranges from the transcript_segments keys
    timestamp_ranges = [
    parse_timestamp_range(ts)
    for ts in transcript_segments.keys()
]
    # Extract features

    # if there's no transcript segments, we return empty array of shape (0, num_prosody_features = 13)
    if not timestamp_ranges:
        return np.empty((0, 13))  # Assuming 13 prosody features as per the example
    
    feature_vector_list = analyze_prosody(video_path, timestamp_ranges)

    # Convert list of vectors → proper 2D numpy array
    feature_vector_array = np.vstack(feature_vector_list)

    
    return feature_vector_array

In [6]:
# Load RecruitView dataset and parse transcripts by timestamp (one row per participant)

dataset = load_dataset("AI4A-lab/RecruitView")
train = dataset["train"]

In [23]:

# Use personality_score from dataset if present, else placeholder
personality_col = None
for col in ("personality_score", "overall_personality", "personality"):
    if col in train.column_names:
        personality_col = col
        break

# Build table: one row per participant; transcript_segments = { "0:01 - 0:11": "words", ... }
rows = []
for participant_id in range(len(train)):

    # --- START I/O Section ---
    # INPUTS from RecruitView Dataset
    transcript_str = train["transcript"][participant_id]
    video_file = train["video"][participant_id]
    video_path = video_file._hf_encoded['path']

    # OUTPUT 
    score = train[personality_col][participant_id] if personality_col else None

    # --- END I/O Section ---


    # --- START Parsing Section ---

    # Parsing Transcript 
    segments = parse_transcript(transcript_str)
    transcript_segments = {ts: text for ts, text in segments}

    # Parsing Audio to Get Prosody Features
    prosodyFeaturesArray = parse_audio_return_prosody_features(video_path, transcript_segments)
    
    # --- END Parsing Section ---

    
    rows.append({
        "participant_id": participant_id,
        "transcript_segments": transcript_segments,
        "personality_score": score,
        "prosody_features" : prosodyFeaturesArray
    })

table = pd.DataFrame(rows)
print(f"Loaded {len(train)} participants (one row each).")
print("Table columns:", list(table.columns))
print("Example transcript_segments (first participant):", table["transcript_segments"].iloc[0])
table.head()

/var/folders/hw/_mprf1z97jsbbjskdfz9419w0000gn/T/ipykernel_22873/4090398377.py:118: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, rate = librosa.load(audio_path, sr=None)


EOFError: 

In [ ]:
# Embed segment transcripts: one 2D array per participant (n_segments, embed_dim)
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# One list of segment texts per participant (order preserved from dict)
segment_texts_per_participant = [list(row["transcript_segments"].values()) for _, row in table.iterrows()]
# Flatten to encode all segments in one batch
all_segment_texts = [t for segs in segment_texts_per_participant for t in segs]
embeddings_flat = model.encode(all_segment_texts, show_progress_bar=True)

# Split back into 2D arrays: one (n_segments, embed_dim) per participant
sizes = [len(segs) for segs in segment_texts_per_participant]
splits = np.cumsum(sizes)[:-1]
table["transcript_embeddings"] = np.split(embeddings_flat, splits)

print(f"Total segments: {len(all_segment_texts)}. Embedding dim: {embeddings_flat.shape[1]}.")
print("Per-participant shapes (n_segments, embed_dim):", [e.shape for e in table["transcript_embeddings"].iloc[:3]])
table.head()

In [ ]:
print(table.head())